In [839]:
import pandas as pd
import numpy as np


In [840]:
df=pd.read_excel('Final.xlsx')
# df

In [841]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53817 entries, 0 to 53816
Data columns (total 7 columns):
 #   Column                                                  Non-Null Count  Dtype 
---  ------                                                  --------------  ----- 
 0   Job Title                                               53817 non-null  object
 1   Company                                                 52738 non-null  object
 2   Location (City / State / Country)                       53500 non-null  object
 3   Salary (if available)                                   29110 non-null  object
 4   Job Type (Remote / Hybrid / On-site)                    53817 non-null  object
 5   Skills mentioned in the job description (if available)  40910 non-null  object
 6   Post Date                                               53571 non-null  object
dtypes: object(7)
memory usage: 2.9+ MB


In [842]:
df = df.dropna(subset=['Job Title'])

In [843]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53817 entries, 0 to 53816
Data columns (total 7 columns):
 #   Column                                                  Non-Null Count  Dtype 
---  ------                                                  --------------  ----- 
 0   Job Title                                               53817 non-null  object
 1   Company                                                 52738 non-null  object
 2   Location (City / State / Country)                       53500 non-null  object
 3   Salary (if available)                                   29110 non-null  object
 4   Job Type (Remote / Hybrid / On-site)                    53817 non-null  object
 5   Skills mentioned in the job description (if available)  40910 non-null  object
 6   Post Date                                               53571 non-null  object
dtypes: object(7)
memory usage: 2.9+ MB


In [844]:
df[['salary_min', 'salary_max']] = df['Salary (if available)'].str.extract(
    r'(\d+)\D+(\d+)'
)
df = df.drop(columns=['Salary (if available)'])
df

,Job Title,Company,Location (City / State / Country),Job Type (Remote / Hybrid / On-site),Skills mentioned in the job description (if available),Post Date,salary_min,salary_max
0,Senior Supply Chain Analyst,Atrium Health,"Charlotte, NC, US",On-site,"SQL, Python, Excel, Tableau, Power BI, Snowfla...",2026-05-29,41,61
1,Data Analyst - G126 - Fire/EMS,Columbus Consolidated Government,"Columbus, GA, US",On-site,"SQL, Excel, Power BI, statistics, data visuali...",2026-05-29,59494,68972
2,Senior Lifecycle Marketing Specialist,BlackLine,"Los Angeles, CA, US",On-site,A/B testing,2026-05-09,95000,119000
3,Internal Audit Specialist - Bilingual,Nissin Foods,"Torrance, CA, US",On-site,"Excel, Power BI",2026-05-08,80000,100000
4,Research Data Analyst 1,Case Western Reserve University,"Cleveland, OH, US",On-site,"R, machine learning, data visualization",2026-05-29,52705,66672
...,...,...,...,...,...,...,...,...
53812,Project Controls | Capital Projects | Cost Ana...,AES Corporation,"Indianapolis, IN, US",On-site,"Power BI, SAP",NaN,NaN,NaN
53813,Analyste stratégie de prix,Bumper to Bumper Canada,"Boucherville, Quebec, Canada",On-site,NaN,NaN,NaN,NaN
53814,Business Planner - HoMES,Toronto Community Housing,"Toronto, Ontario, Canada",On-site,NaN,NaN,NaN,NaN
53815,Financial Analyst,Acquird.io,"Toronto, Ontario, Canada",On-site,NaN,NaN,NaN,NaN


In [845]:
location_split = df['Location (City / State / Country)'].str.split(',', expand=True)

print(location_split.head())
print(location_split.shape)

             0    1    2     3
0    Charlotte   NC   US  None
1     Columbus   GA   US  None
2  Los Angeles   CA   US  None
3     Torrance   CA   US  None
4    Cleveland   OH   US  None
(53817, 4)


In [846]:
df[['City','State','Country']] = location_split.iloc[:, :3]
df = df.drop(columns=['Location (City / State / Country)'])


In [847]:
# df

In [848]:

df['salary_min'] = pd.to_numeric(df['salary_min'], errors='coerce')
print(df['salary_min'].dtype)

df['salary_max']=pd.to_numeric(df['salary_max'], errors='coerce')
print(df['salary_max'].dtype)

float64
float64


In [849]:
df['salary_min'].describe()

,salary_min
count,29106.000000
mean,101481.989315
std,53385.029943
min,0.000000
25%,71546.000000
50%,100000.000000
75%,134500.000000
max,550000.000000


In [850]:
def convert_salary(row):
  if row ['salary_min'] <100:
    return row['salary_min']*160
  elif row['salary_min'] < 9800:
    return row['salary_min']*12
  else:
    return row['salary_min']


In [851]:
df['salary_min'] = df.apply(convert_salary, axis=1)
df['salary_min'].min()


0.0

In [852]:
df['salary_min'] = pd.to_numeric(df['salary_min'], errors='coerce')
df['salary_min'] = df['salary_min'].mask(df['salary_min'] <= 0, np.nan)

median_val = df['salary_min'].median()
df['salary_min'] = df['salary_min'].fillna(median_val)

print(f"New Min: {df['salary_min'].min()}")
print("Min:", df['salary_min'].min())
print("Median used:", median_val)

New Min: 160.0
Min: 160.0
Median used: 100000.0


In [853]:
threshold = 1000
df['salary_min'] = df['salary_min'].mask(df['salary_min'] < threshold, np.nan)
df['salary_min'] = df['salary_min'].fillna(df['salary_min'].median())
df['salary_min'].min()

1120.0

In [854]:
df['salary_min'].isna().sum()

np.int64(0)

In [855]:

df['salary_min'].describe()

,salary_min
count,53817.000000
mean,101423.889217
std,38073.744664
min,1120.000000
25%,96000.000000
50%,100000.000000
75%,105000.000000
max,550000.000000


In [856]:
def convert_salaryM(row):
  if row ['salary_max'] < 505:
    return row['salary_max']*160
  elif row['salary_max'] <12000:
    return row['salary_max']*12
  else:
    return row['salary_max']


In [857]:
df['salary_max'] = df.apply(convert_salaryM, axis=1)
df['salary_max'].min()

0.0

In [858]:
df['salary_max'] = pd.to_numeric(df['salary_max'], errors='coerce')
df['salary_max'] = df['salary_max'].mask(df['salary_max'] <= 0, np.nan)

median_val = df['salary_max'].median()
df['salary_max'] = df['salary_max'].fillna(median_val)

print(f"New Min: {df['salary_min'].min()}")
print("Min:", df['salary_min'].min())
print("Median used:", median_val)

New Min: 1120.0
Min: 1120.0
Median used: 159100.0


In [859]:
median_val = df['salary_max'].median()
df['salary_min'] = df['salary_max'].fillna(median_val)

print(f"New Max: {df['salary_max'].min()}")
print("Max:", df['salary_max'].min())
print("Median used:", median_val)

New Max: 160.0
Max: 160.0
Median used: 159100.0


In [860]:
df['salary_max']

,salary_max
0,9760.0
1,68972.0
2,119000.0
3,100000.0
4,66672.0
...,...
53812,159100.0
53813,159100.0
53814,159100.0
53815,159100.0


In [861]:
from collections import Counter

skill_column = 'Skills mentioned in the job description (if available)'

all_skills = []

for skills in df[skill_column]:
    if pd.notna(skills):
        skill_list = [s.strip().lower() for s in str(skills).split(',') if s.strip()]
        all_skills.extend(skill_list)


skill_counts = Counter(all_skills)

result_df = pd.DataFrame(skill_counts.items(), columns=['Skill', 'Count'])
result_df = result_df.sort_values('Count', ascending=False).reset_index(drop=True)


result_df.to_csv('skills_for_powerbi.csv', index=False, encoding='utf-8-sig')
result_df.to_excel('skills_for_powerbi.xlsx', index=False)  # لو بتفضلي Excel

print(f"uniqe skills {len(result_df)}")

print(result_df.head(10).to_string(index=False))


uniqe skills 74
           Skill  Count
           excel  13187
          python  11217
             sql  11184
             aws   7158
machine learning   6911
           azure   6748
           agile   6286
        power bi   5718
         tableau   4999
      recruiting   4916


In [862]:
df['Avrage']=(df['salary_max']+df['salary_min'])/2

In [863]:
df

,Job Title,Company,Job Type (Remote / Hybrid / On-site),Skills mentioned in the job description (if available),Post Date,salary_min,salary_max,City,State,Country,Avrage
0,Senior Supply Chain Analyst,Atrium Health,On-site,"SQL, Python, Excel, Tableau, Power BI, Snowfla...",2026-05-29,9760.0,9760.0,Charlotte,NC,US,9760.0
1,Data Analyst - G126 - Fire/EMS,Columbus Consolidated Government,On-site,"SQL, Excel, Power BI, statistics, data visuali...",2026-05-29,68972.0,68972.0,Columbus,GA,US,68972.0
2,Senior Lifecycle Marketing Specialist,BlackLine,On-site,A/B testing,2026-05-09,119000.0,119000.0,Los Angeles,CA,US,119000.0
3,Internal Audit Specialist - Bilingual,Nissin Foods,On-site,"Excel, Power BI",2026-05-08,100000.0,100000.0,Torrance,CA,US,100000.0
4,Research Data Analyst 1,Case Western Reserve University,On-site,"R, machine learning, data visualization",2026-05-29,66672.0,66672.0,Cleveland,OH,US,66672.0
...,...,...,...,...,...,...,...,...,...,...,...
53812,Project Controls | Capital Projects | Cost Ana...,AES Corporation,On-site,"Power BI, SAP",NaN,159100.0,159100.0,Indianapolis,IN,US,159100.0
53813,Analyste stratégie de prix,Bumper to Bumper Canada,On-site,NaN,NaN,159100.0,159100.0,Boucherville,Quebec,Canada,159100.0
53814,Business Planner - HoMES,Toronto Community Housing,On-site,NaN,NaN,159100.0,159100.0,Toronto,Ontario,Canada,159100.0
53815,Financial Analyst,Acquird.io,On-site,NaN,NaN,159100.0,159100.0,Toronto,Ontario,Canada,159100.0


In [864]:
df['Country'] = df['Country'].replace('', np.nan)
df['Country'] = df['Country'].fillna(df['State'])

In [865]:
df['Country'].unique()

array([' US', ' USA timezones', None, ' Ontario', ' United States',
       ' Washington', ' Florida', ' CA', ' Georgia', ' New York',
       ' California', ' Tennessee', ' South Carolina', '', ' New Mexico',
       ' Pennsylvania', ' Canada', nan, ' OH', ' GA', ' NY', ' VA', ' IL',
       ' AL', ' FL', ' TX', ' PA', ' MA', ' WV', ' MN', ' DE', ' WA',
       ' NJ', ' IA', ' MT', ' NC', ' TN', ' WI', ' IN', ' UT', ' CO',
       ' MD', ' AR', ' MI', ' MO', ' CT', ' OR', ' AZ', ' ND', ' HI',
       ' KS', ' NE', ' MS', ' NV', ' KY', ' LA', ' OK', ' ID', ' RI',
       ' SC', ' NH', ' NM'], dtype=object)

In [866]:
def map(x):
    x = str(x).strip()

    us_states = [
        'US','USA timezones','United States','CA','NY','TX','FL','WA','OH','GA','PA','IL',
        'VA','NC','SC','TN','MA','NJ','MI','IN','AZ','CO','OR','NV','UT','MN','MO','WI',
        'IA','KS','OK','LA','AR','MS','AL','KY','NE','ID','MT','ND','SD','CT','RI','DE',
        'NH','NM','WV','HI','Washington','Florida','Georgia','New York','California',
        'Tennessee','South Carolina','New Mexico','USA (Remote)','United States (Remote)',"USA"
    ]

    canada = ['Canada','Ontario','Canada (Remote)']

    if x in us_states:
        return "US"
    elif x in canada:
        return "Canada"
    else:
        x


In [867]:
df['Country'] = df['Country'].apply(map)

In [868]:


df['Country'].unique()

array(['US', None, 'Canada'], dtype=object)

In [869]:
df

,Job Title,Company,Job Type (Remote / Hybrid / On-site),Skills mentioned in the job description (if available),Post Date,salary_min,salary_max,City,State,Country,Avrage
0,Senior Supply Chain Analyst,Atrium Health,On-site,"SQL, Python, Excel, Tableau, Power BI, Snowfla...",2026-05-29,9760.0,9760.0,Charlotte,NC,US,9760.0
1,Data Analyst - G126 - Fire/EMS,Columbus Consolidated Government,On-site,"SQL, Excel, Power BI, statistics, data visuali...",2026-05-29,68972.0,68972.0,Columbus,GA,US,68972.0
2,Senior Lifecycle Marketing Specialist,BlackLine,On-site,A/B testing,2026-05-09,119000.0,119000.0,Los Angeles,CA,US,119000.0
3,Internal Audit Specialist - Bilingual,Nissin Foods,On-site,"Excel, Power BI",2026-05-08,100000.0,100000.0,Torrance,CA,US,100000.0
4,Research Data Analyst 1,Case Western Reserve University,On-site,"R, machine learning, data visualization",2026-05-29,66672.0,66672.0,Cleveland,OH,US,66672.0
...,...,...,...,...,...,...,...,...,...,...,...
53812,Project Controls | Capital Projects | Cost Ana...,AES Corporation,On-site,"Power BI, SAP",NaN,159100.0,159100.0,Indianapolis,IN,US,159100.0
53813,Analyste stratégie de prix,Bumper to Bumper Canada,On-site,NaN,NaN,159100.0,159100.0,Boucherville,Quebec,Canada,159100.0
53814,Business Planner - HoMES,Toronto Community Housing,On-site,NaN,NaN,159100.0,159100.0,Toronto,Ontario,Canada,159100.0
53815,Financial Analyst,Acquird.io,On-site,NaN,NaN,159100.0,159100.0,Toronto,Ontario,Canada,159100.0


In [870]:
# df.to_csv("df.csv", index=False)
# from google.colab import files
# files.download("df.csv")

In [871]:
DA_keywords = [
    'data analyst',
    'reporting analyst',
    'sql analyst',
    'analysis'
]



In [872]:
BI_keywords = [
    'bi analyst',
    'business intelligence',
    'business intelligence analyst',
    'bi developer',
    'power bi',
    'power bi developer',
    'power bi analyst',
    'mis analyst',
    'mis executive',
    'business analyst'
]


In [873]:
AI_keywords = [
    'ai',
    'artificial intelligence',
    'ai engineer',
    'generative ai',
    'gen ai',
    'deep learning',
    'nlp',
    'ai developer',
    'ai specialist'
]


In [874]:
keywords = [
    'it support',
    'help desk',
    'helpdesk',
    'technical support',
    'desktop support',
    'service desk',
    'it specialist',
    'it analyst',
    'it administrator',
    'system administrator',
    'systems administrator',
    'network administrator',
    'network engineer',
    'it engineer',
    'infrastructure engineer',
    'it coordinator',
    'it technician',
    'technical analyst',
    'application support',
    'support engineer'
]


In [875]:
df_AI = df[df['Job Title'].str.lower().str.contains('|'.join(AI_keywords), na=False)].copy()
df_BI = df[df['Job Title'].str.lower().str.contains('|'.join(BI_keywords), na=False)].copy()
df_DA = df[df['Job Title'].str.lower().str.contains('|'.join(DA_keywords), na=False)].copy()
df_IT = df[df['Job Title'].str.lower().str.contains('|'.join(keywords), na=False)].copy()

In [876]:
df_AI.to_csv('AI_Jobs_Cleaned.csv', index=False)
df_BI.to_csv('BI_Jobs_Cleaned.csv', index=False)
df_DA.to_csv('DA_Jobs_Cleaned.csv', index=False)
df_IT.to_csv('IT_Jobs_Cleaned.csv', index=False)


print(f"AI jobs num{len(df_AI)}")
print(f"BI jobs num:{len(df_BI)}")
print(f"DA jobs num{len(df_DA)}")
print(f"IT jobs num{len(df_IT)}")

AI jobs num6823
BI jobs num:2674
DA jobs num1990
IT jobs num1043


In [877]:
# df_IT.info()

In [878]:
df['df_BI'] = df['Job Title'].str.lower().str.contains('|'.join(BI_keywords), na=False)
df['df_AI'] = df['Job Title'].str.lower().str.contains('|'.join(AI_keywords), na=False)
df['df_Analysis'] = df['Job Title'].str.lower().str.contains('|'.join(DA_keywords), na=False)

In [879]:
df_DA

,Job Title,Company,Job Type (Remote / Hybrid / On-site),Skills mentioned in the job description (if available),Post Date,salary_min,salary_max,City,State,Country,Avrage
1,Data Analyst - G126 - Fire/EMS,Columbus Consolidated Government,On-site,"SQL, Excel, Power BI, statistics, data visuali...",2026-05-29,68972.0,68972.0,Columbus,GA,US,68972.0
4,Research Data Analyst 1,Case Western Reserve University,On-site,"R, machine learning, data visualization",2026-05-29,66672.0,66672.0,Cleveland,OH,US,66672.0
21,"Senior Associate, Mission Support (Planning) -...",Team Rubicon,On-site,"Power BI, data visualization, business intelli...",2026-05-30,85630.0,85630.0,New York,NY,US,85630.0
22,"Senior Associate, Mission Support (Planning) -...",Team Rubicon,On-site,"Power BI, data visualization, business intelli...",2026-05-30,85630.0,85630.0,New Orleans,LA,US,85630.0
23,"Senior Associate, Mission Support (Planning) -...",Team Rubicon,On-site,"Power BI, data visualization, business intelli...",2026-05-30,85630.0,85630.0,Los Angeles,CA,US,85630.0
...,...,...,...,...,...,...,...,...,...,...,...
53789,"Data Analyst, New Grad (Canada)",Jobright.ai,On-site,NaN,NaN,159100.0,159100.0,NaN,NaN,None,159100.0
53791,Financial Planning & Analysis (FP&A) Volume & ...,Stellantis,On-site,"Power BI, business intelligence, forecasting",NaN,159100.0,159100.0,Auburn Hills,MI,US,159100.0
53797,Senior Data Analyst,Applicantz,On-site,NaN,NaN,159100.0,159100.0,Toronto,Ontario,Canada,159100.0
53801,IT Data Analyst,InSync Systems,On-site,NaN,NaN,159100.0,159100.0,Calgary,Alberta,Canada,159100.0


In [880]:
df_BI

,Job Title,Company,Job Type (Remote / Hybrid / On-site),Skills mentioned in the job description (if available),Post Date,salary_min,salary_max,City,State,Country,Avrage
27,Business Intelligence Developer III,Sutter Health,On-site,"SQL, Excel, Power BI, statistics, business int...",2026-05-29,12000.0,12000.0,Sacramento,CA,US,12000.0
170,Business Analyst,Tata Consultancy Services (TCS),On-site,"SQL, Python, Tableau, Power BI, SAS, ETL, Agil...",2026-05-29,100000.0,100000.0,Cleveland,OH,US,100000.0
316,Principal HRIS Business Analyst,Marvell Technology,On-site,"generative AI, Jira, Agile",2026-05-29,208000.0,208000.0,Santa Clara,CA,US,208000.0
390,"Senior Business Analyst, Liquidity, Market, an...",Capital One,On-site,"SQL, statistics",2026-05-29,126900.0,126900.0,McLean,VA,US,126900.0
391,Product Marketing Business Analyst,"FormFactor, Inc.",On-site,"SQL, Excel, Tableau, Power BI, Oracle, data vi...",2026-05-29,111300.0,111300.0,Livermore,CA,US,111300.0
...,...,...,...,...,...,...,...,...,...,...,...
53763,Business Analyst,OMNISTARR,On-site,NaN,NaN,159100.0,159100.0,Toronto,Ontario,Canada,159100.0
53764,Business Analyst (Fixed Term),Aligned Capital Partners Inc.,On-site,NaN,NaN,159100.0,159100.0,Burlington,Ontario,Canada,159100.0
53768,Business Intelligence Analyst,EF Educational Tours,On-site,NaN,NaN,159100.0,159100.0,Toronto,Ontario,Canada,159100.0
53769,Business Analyst -PL/SQL/T-SQL,J&M Group,On-site,NaN,NaN,159100.0,159100.0,Toronto,Ontario,Canada,159100.0


In [881]:
df_AI

,Job Title,Company,Job Type (Remote / Hybrid / On-site),Skills mentioned in the job description (if available),Post Date,salary_min,salary_max,City,State,Country,Avrage
0,Senior Supply Chain Analyst,Atrium Health,On-site,"SQL, Python, Excel, Tableau, Power BI, Snowfla...",2026-05-29,9760.0,9760.0,Charlotte,NC,US,9760.0
9,Email Marketing Specialist,Pepperdine University,Hybrid,A/B testing,2026-05-08,4480.0,4480.0,Calabasas,CA,US,4480.0
12,Sr. Product Manager – AI & Intelligent Orderin...,Tillster,On-site,"machine learning, NLP, Agile, Scrum, predictiv...",2026-05-14,165000.0,165000.0,Los Angeles,CA,US,165000.0
17,Campaign Marketing Specialist,VoteJesusPatino,On-site,SQL,2026-05-12,2880.0,2880.0,Tustin,CA,US,2880.0
34,"Paid Advertising, SEO & Email Marketing Specia...",MIA SECRET,On-site,A/B testing,2026-05-15,55000.0,55000.0,City of Industry,CA,US,55000.0
...,...,...,...,...,...,...,...,...,...,...,...
53799,AI and Data - Senior Consultant - Data Scientist,EY,On-site,NaN,NaN,159100.0,159100.0,St John’s,Newfoundland and Labrador,Canada,159100.0
53802,"Analyst/Consultant, AI/ML Models - Financial E...",Deloitte,On-site,NaN,NaN,159100.0,159100.0,Toronto,Ontario,Canada,159100.0
53803,Machine Learning Engineer III / Senior Machine...,Workday,On-site,NaN,NaN,159100.0,159100.0,Calgary,Alberta,Canada,159100.0
53807,Technology Analyst - GenAI,Infosys,On-site,NaN,NaN,159100.0,159100.0,Mississauga,Ontario,Canada,159100.0


In [882]:
df_IT

,Job Title,Company,Job Type (Remote / Hybrid / On-site),Skills mentioned in the job description (if available),Post Date,salary_min,salary_max,City,State,Country,Avrage
3,Internal Audit Specialist - Bilingual,Nissin Foods,On-site,"Excel, Power BI",2026-05-08,100000.0,100000.0,Torrance,CA,US,100000.0
48,Fixed Income Corporate Credit Analyst,Principal Financial Group,On-site,"Python, Excel, generative AI, statistics",2026-04-24,191000.0,191000.0,Des Moines,IA,US,191000.0
59,Remote Sensing Technical Support Advisor in Ve...,Southern California Edison,Hybrid,"data governance, business intelligence",2026-05-18,171600.0,171600.0,Pomona,CA,US,171600.0
299,Quality Review and Audit Analyst - Cigna Healt...,The Cigna Group,On-site,Excel,2026-05-29,97400.0,97400.0,Remote,US,US,97400.0
443,Systems Administrator (ERP),SCS Engineers,On-site,"SQL, Power BI, business intelligence",2026-05-29,110000.0,110000.0,Greenwood Village,CO,US,110000.0
...,...,...,...,...,...,...,...,...,...,...,...
52515,"Machine Learning Engineer, Customer Support En...",Airbnb,On-site,NaN,2026-05-28,159100.0,159100.0,NaN,NaN,None,159100.0
52669,Credit Analyst I,Bank of Ann Arbor,On-site,NaN,2026-04-21,159100.0,159100.0,Ann Arbor,MI,US,159100.0
53096,Circuit Analyst,TechInsights,On-site,NaN,2026-05-05,159100.0,159100.0,Ottawa,ON,US,159100.0
53196,IT Analyst,Solar Turbines,On-site,NaN,2026-05-26,159100.0,159100.0,San Diego,CA,US,159100.0


In [887]:
df_BI.to_csv("df_DA.csv", index=False)
from google.colab import files
files.download("df_BI.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>